In [ ]:
import vegas
import numpy as np

# --- THE FIX: Tell VEGAS to send 2D batches instead of 1D points ---
@vegas.batchintegrand
def target_pdf(x):
    # x is now guaranteed to be a batch of coordinates of shape (N_samples, 2)
    norm = 1.0 / (2.0 * np.pi * 0.2)
    
    # Peak 1 at (2, 2)
    diff1_sq = (x[:, 0] - 2.0)**2 + (x[:, 1] - 2.0)**2
    pdf1 = norm * np.exp(-diff1_sq / (2.0 * 0.2))
    
    # Peak 2 at (-2, -2)
    diff2_sq = (x[:, 0] + 2.0)**2 + (x[:, 1] + 2.0)**2
    pdf2 = norm * np.exp(-diff2_sq / (2.0 * 0.2))
    
    return 0.5 * pdf1 + 0.5 * pdf2

# The rest remains exactly the same
integ = vegas.Integrator([[-10.0, 10.0], [-10.0, 10.0]])

print("Starting VEGAS Adaptation with high statistics...")
integ(target_pdf, nitn=20, neval=100000)

print("Starting VEGAS Final Integration with massive statistics...")
# Runs 10 separate batches of 500,000 points, and averages them!
result = integ(target_pdf, nitn=10, neval=500000)

print("\n--- FINAL RESULTS ---")
print(result.summary())

integral = result.mean
error = result.sdev
variance = (error**2) * 50000

print(f"\nCalculated Integral : {integral:.6f} +/- {error:.6f}")
print(f"True Math Integral  : 1.000000")
print(f"Variance (sigma^2)  : {variance:.6f}")

In [ ]:
import vegas
import numpy as np

# Tell VEGAS to send us 2D batches for fast NumPy math
@vegas.batchintegrand
def target_pdf_vegas(x):
    # x[:, 0] is Energy (E) natively sampled from 50 to 150
    # x[:, 1] is cos(theta) natively sampled from -1 to 1
    E = x[:, 0]
    cos_theta = x[:, 1]
    
    m_Z = 91.1876
    Gamma_Z = 2.4952
    
    # The exact same Matrix Element math
    s = E**2
    bw_numerator = s
    bw_denominator = (s - m_Z**2)**2 + (m_Z * Gamma_Z)**2
    breit_wigner = bw_numerator / bw_denominator
    
    angular = 1.0 + cos_theta**2
    
    # Return the raw physics probability (NOT log-space)
    return breit_wigner * angular

# ==========================================
# SET UP VEGAS WITH EXACT PHYSICAL LIMITS
# ==========================================
# [Energy Range], [Angle Range]
integ = vegas.Integrator([[50.0, 150.0], [-1.0, 1.0]])

# ==========================================
# TRAINING AND INTEGRATION
# ==========================================
print("Starting VEGAS Adaptation (Z Boson)...")
integ(target_pdf_vegas, nitn=30, neval=1000000)

print("\nStarting VEGAS Final Integration...")
# Using 10 iterations of 500,000 points to crush the error bar
result = integ(target_pdf_vegas, nitn=30, neval=1000000)

print("\n--- FINAL RESULTS ---")
print(result.summary())

integral = result.mean
error = result.sdev

# Total points = 10 iterations * 1,000,000 points per iteration
total_samples = 30 * 1000000 

# Calculate standard variance to compare with the PyTorch model
variance = (error**2) * total_samples

print(f"\nCalculated Integral : {integral:.6f} +/- {error:.6f}")
print(f"Variance (sigma^2)  : {variance:.6f}")

In [ ]:
import scipy.integrate as integrate

m_Z = 91.1876
Gamma_Z = 2.4952

# 1. Define only the Energy portion of the Breit-Wigner
def energy_integrand(E):
    s = E**2
    bw_numerator = s
    bw_denominator = (s - m_Z**2)**2 + (m_Z * Gamma_Z)**2
    return bw_numerator / bw_denominator

# 2. Integrate with SciPy Quadrature between exactly 50 and 150
energy_integral, _ = integrate.quad(energy_integrand, 50.0, 150.0)

# 3. The exact analytical solution for the angular integral (-1 to 1)
angular_integral = 8.0 / 3.0  # (which is 2.6666...)

# 4. Multiply them back together
total_exact_integral = energy_integral * angular_integral

print(f"Energy Integral (50 to 150) : {energy_integral:.6f}")
print(f"Total Exact Integral        : {total_exact_integral:.6f}")

In [ ]:
import vegas
import numpy as np

# 1. Define the exact same 8D target function (in standard NumPy)
def rosenbrock_pdf(x):
    # x is a single 8-dimensional point: [x0, x1, x2, x3, x4, x5, x6, x7]
    x_np = np.asarray(x)
    
    # We slice the array just like we did in PyTorch
    x_i = x_np[:-1]    # The first 7 dimensions
    x_next = x_np[1:]  # The next 7 dimensions
    
    term1 = 100.0 * (x_next - x_i**2)**2
    term2 = (1.0 - x_i)**2
    
    # We use the exact same -0.05 scaling factor inside the exponent
    f_phys = np.exp(-0.05 * np.sum(term1 + term2))
    
    return f_phys

# 2. Initialize the VEGAS Integrator
# We tell it we have 8 dimensions, all bounded between -5.0 and +5.0
bounds = [[-5.0, 5.0]] * 8
integ = vegas.Integrator(bounds)

print("Step 1: Training the VEGAS grid (Warm-up)...")
# We give it 10 iterations of 100,000 points just to try and find the 8D ribbon
integ(rosenbrock_pdf, nitn=30, neval=1000000)

print("\nStep 2: Computing the final 8D integral...")
# We blast it with half a million points per iteration to get the final number
result = integ(rosenbrock_pdf, nitn=30, neval=1000000)

# 3. Print the brutal reality
print("\n=========================================")
print("          FINAL VEGAS RESULTS            ")
print("=========================================")
print(result.summary())
print(f"\nCalculated Integral : {result.mean:.6f} +/- {result.sdev:.6f}")
print(f"Relative Error      : {(result.sdev / result.mean) * 100:.2f} %")
print("=========================================")

In [2]:
import vegas
import numpy as np

# 1. Define the 16D Topological Trap in pure NumPy
def topological_trap_pdf(x):
    # x is a single 16-dimensional point
    x_np = np.asarray(x)
    sigma = 0.2
    
    # Calculate the squared distance to Galaxy A (+2.5 on all axes)
    dist_A = np.sum((x_np - 2.5)**2)
    
    # Calculate the squared distance to Galaxy B (-2.5 on all axes)
    dist_B = np.sum((x_np + 2.5)**2)
    
    # VEGAS needs the raw probability (not the log probability)
    prob_A = np.exp(-0.5 * dist_A / sigma**2)
    prob_B = np.exp(-0.5 * dist_B / sigma**2)
    
    return prob_A + prob_B

# 2. Initialize VEGAS for 16 Dimensions
bounds = [[-5.0, 5.0]] * 16
integ = vegas.Integrator(bounds)

print("Step 1: VEGAS attempting to find the 16D galaxies (Warm-up)...")
# We give it 10 iterations of 100,000 darts to map the space
integ(topological_trap_pdf, nitn=10, neval=1000000)

print("\nStep 2: Computing the final 16D integral...")
# We blast it with half a million darts per iteration for the final test
result = integ(topological_trap_pdf, nitn=10, neval=5000000)

# 3. Print the brutal reality
print("\n=========================================")
print("          FINAL VEGAS RESULTS            ")
print("=========================================")
print(result.summary())
# Using scientific notation (.10e) because the numbers might be microscopically small
print(f"\nCalculated Integral : {result.mean:.10e} +/- {result.sdev:.10e}")
print("=========================================")

Step 1: VEGAS attempting to find the 16D galaxies (Warm-up)...

Step 2: Computing the final 16D integral...

          FINAL VEGAS RESULTS            
itn   integral        wgt average     chi2/dof        Q
-------------------------------------------------------
  1   1.394(73)e-05   1.394(73)e-05       0.00     1.00
  2   1.64(11)e-05    1.471(61)e-05       3.59     0.06
  3   1.539(33)e-05   1.524(29)e-05       2.29     0.10
  4   1.598(16)e-05   1.581(14)e-05       3.24     0.02
  5   1.5953(76)e-05  1.5920(66)e-05      2.64     0.03
  6   1.5930(42)e-05  1.5927(36)e-05      2.11     0.06
  7   1.5931(25)e-05  1.5930(21)e-05      1.76     0.10
  8   1.5909(17)e-05  1.5917(13)e-05      1.60     0.13
  9   1.5924(12)e-05  1.59209(88)e-05     1.42     0.18
 10   1.59059(92)e-05 1.59137(64)e-05     1.41     0.18


Calculated Integral : 1.5913684992e-05 +/- 6.3855651431e-09
